> **Sentetik veri.** Bu defter `python -m sentetik.uret` ile üretilen uydurma veriyle çalışır. Sayılar gerçek bir şirkete ait değildir ve gerçek projenin sonuçlarını temsil etmez; amaç yöntemi uçtan uca göstermektir. *(Synthetic data: numbers are not real results.)*

# Sinyal — 01 · Keşifsel Veri Analizi (Model A tablosu)

**Kural:** EDA yalnızca **eğitim** kesimleri (2024-10-01 → 2025-07-01) üzerinde yapılır. Test kesimi (2025-10-01) bu defterde **hiç incelenmez**; tampon kesimler yalnızca etiket oranı grafiğinde görünür.

**Gizlilik:** Çıktılar toplu istatistiktir. Ham satır, musteri_id ve TL tutarlar paylaşılmaz; tutarlar endeks (`_e`) olarak gösterilir.

**Amaç:** (1) eksiklik ve kayma kontrolü, (2) etiketle ilişkili özellikleri görmek, (3) gereksiz/tekrarlı özellikleri ayıklamak, (4) tanımları kesinleştirmek için kanıt toplamak.

In [ ]:
import sys
from pathlib import Path

KOK = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(KOK))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.veri import tablo_yukle
from src.ozellikler import endeksle, model_a_ozellikleri, PARASAL

FIG = KOK / "notebooks" / "figurler"
FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.3f}".format)

df = endeksle(tablo_yukle("egitim_a"))          # ilk çalıştırmada BigQuery'den çeker
egitim = df[df["kume"] == "egitim"].copy()
ozellikler = model_a_ozellikleri(df)
sayisal = [c for c in ozellikler if pd.api.types.is_numeric_dtype(df[c])]

print("Toplam:", df.shape, "| Eğitim:", egitim.shape)
print("Özellik sayısı:", len(ozellikler), "| sayısal:", len(sayisal))

## 1. Genel bakış ve eksiklik (eğitim kesimleri)

In [ ]:
eksik = (egitim[ozellikler].isna().mean()
         .rename("eksik_oran").to_frame()
         .assign(tip=lambda t: [str(egitim[c].dtype) for c in t.index])
         .sort_values("eksik_oran", ascending=False))
eksik

## 2. Etiket oranı ve popülasyon (eğitim + tampon)
Mevsimsellik: Kasım–Ocak kesimlerinde etiket oranı yükselir.

In [ ]:
ozet = (df[df["kume"].isin(["egitim", "tampon"])]
        .groupby(["kesim", "kume"])
        .agg(populasyon=("musteri_id", "size"), etiket_orani=("y_etiket", "mean"))
        .reset_index())
display(ozet)

fig, ax = plt.subplots(figsize=(10, 4))
renk = ozet["kume"].map({"egitim": "tab:blue", "tampon": "tab:gray"})
ax.bar(ozet["kesim"].dt.strftime("%Y-%m"), ozet["etiket_orani"], color=renk)
ax.set_ylabel("90 günde tekrar alım oranı")
ax.set_title("Kesim bazında etiket oranı (gri = tampon)")
plt.xticks(rotation=45)
plt.tight_layout(); fig.savefig(FIG / "01_etiket_orani.png", dpi=150)

## 3. Enflasyon ve endeksleme
Ham TL medyanı kesimden kesime yükselir; endekslenmiş (`_e`) hali yatay kalmalıdır.

In [ ]:
k = df[df["kume"].isin(["egitim", "tampon"])].groupby("kesim")
tablo = pd.DataFrame({
    "ham_medyan_sepet":      k["ort_sepet_9ay"].median(),
    "endeks_medyan_sepet":   k["ort_sepet_9ay_e"].median(),
    "ham_medyan_birim":      k["ort_birim_fiyat_9ay"].median(),
})
tablo["ham_degisim"] = tablo["ham_medyan_sepet"] / tablo["ham_medyan_sepet"].iloc[0] - 1
tablo["endeks_degisim"] = tablo["endeks_medyan_sepet"] / tablo["endeks_medyan_sepet"].iloc[0] - 1
display(tablo[["ham_degisim", "endeks_degisim", "endeks_medyan_sepet"]])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tablo.index, 1 + tablo["ham_degisim"], marker="o", label="Ham TL (ilk kesim = 1)")
ax.plot(tablo.index, 1 + tablo["endeks_degisim"], marker="o", label="Endekslenmiş (ilk kesim = 1)")
ax.axhline(1, color="gray", lw=0.5); ax.legend(); ax.set_title("Medyan sepet: ham vs endekslenmiş")
plt.tight_layout(); fig.savefig(FIG / "02_enflasyon.png", dpi=150)

## 4. Tek özellik → etiket ilişkisi (onda birlik dilimler)
Her özellik için: dilim başına etiket oranı. Düz bir çizgi = zayıf sinyal; monoton artış/azalış = güçlü sinyal.

In [ ]:
def dilim_tablosu(veri, kolon, q=10):
    x = veri[[kolon, "y_etiket"]].dropna()
    dilim = pd.qcut(x[kolon].rank(method="first"), q, labels=False)
    return (x.groupby(dilim)
             .agg(alt=(kolon, "min"), ust=(kolon, "max"), n=("y_etiket", "size"),
                  etiket_orani=("y_etiket", "mean")))

incelenecek = [
    "gun_son_siparis", "siparis_12ay", "siparis_90g", "ort_siparis_arasi_gun",
    "ort_sepet_9ay_e", "ciro_9ay_e", "ort_birim_fiyat_9ay_e",
    "indirim_derinligi_9ay", "son_indirim_derinligi",
    "uyelik_gun", "yas", "pay_app", "ort_kalem", "farkli_kategori2",
]
genel = egitim["y_etiket"].mean()
fig, eksenler = plt.subplots(4, 4, figsize=(16, 13)); eksenler = eksenler.ravel()
ozet_dilim = {}
for ax, kolon in zip(eksenler, incelenecek):
    t = dilim_tablosu(egitim, kolon)
    ozet_dilim[kolon] = t["etiket_orani"].round(3).tolist()
    ax.plot(t.index + 1, t["etiket_orani"], marker="o")
    ax.axhline(genel, color="gray", ls="--", lw=0.8)
    ax.set_title(kolon, fontsize=9); ax.set_xticks(range(1, 11))
for ax in eksenler[len(incelenecek):]:
    ax.axis("off")
plt.suptitle("Dilim (1 = en düşük değer) başına etiket oranı — kesikli çizgi: genel oran")
plt.tight_layout(); fig.savefig(FIG / "03_dilim_etiket.png", dpi=150)

# Paylaşmak için kompakt özet (yalnızca oranlar)
pd.DataFrame(ozet_dilim, index=[f"D{i}" for i in range(1, 11)]).T

## 5. Tek siparişliler vs tekrar edenler
Popülasyonun büyük kısmı 12 ayda tek sipariş vermiş. Asıl problem: ilk siparişten ikinciye geçiş.

In [ ]:
egitim["grup"] = np.where(egitim["siparis_12ay"] == 1, "tek_siparis", "tekrar_eden")
egitim["yakinlik"] = pd.cut(egitim["gun_son_siparis"], [0, 30, 90, 180, 270, 365],
                            labels=["0-30", "31-90", "91-180", "181-270", "271-365"])
display(egitim.groupby("grup")["y_etiket"].agg(["size", "mean"]))
egitim.pivot_table(index="yakinlik", columns="grup", values="y_etiket",
                   aggfunc="mean", observed=True)

## 6. Korelasyon (Spearman) ve tekrarlı özellikler
|r| > 0,85 olan çiftler aynı bilgiyi taşıyor olabilir; birini bırakmak yorumlamayı kolaylaştırır.

In [ ]:
orneklem = egitim[sayisal].sample(min(200_000, len(egitim)), random_state=42)
kor = orneklem.corr(method="spearman")

ciftler = (kor.where(np.triu(np.ones(kor.shape, dtype=bool), k=1))
              .stack().rename("r").reset_index()
              .assign(mutlak=lambda t: t["r"].abs())
              .query("mutlak > 0.85").sort_values("mutlak", ascending=False))
display(ciftler)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(kor, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(kor))); ax.set_xticklabels(kor.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(kor))); ax.set_yticklabels(kor.columns, fontsize=7)
fig.colorbar(im, shrink=0.7); ax.set_title("Spearman korelasyonu (eğitim, örneklem)")
plt.tight_layout(); fig.savefig(FIG / "04_korelasyon.png", dpi=150)

## 7. Kayma kontrolü (eğitim kesimleri arasında)
Her sayısal özelliğin kesim medyanı, ilk kesime göre. %30'dan büyük değişim → incele.

In [ ]:
medyan = egitim.groupby("kesim")[sayisal].median()
degisim = (medyan / medyan.iloc[0].replace(0, np.nan) - 1)
kayma = (degisim.abs().max().rename("max_mutlak_degisim").to_frame()
         .assign(ilk=medyan.iloc[0], son=medyan.iloc[-1])
         .sort_values("max_mutlak_degisim", ascending=False))
kayma[kayma["max_mutlak_degisim"] > 0.30]